# 02 — Exploratory Data Analysis

Each figure answers a specific question. Figures are written to `outputs/figures/`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')

import pandas as pd, numpy as np
import config
from src import data_loader as dl

import matplotlib.pyplot as plt
import seaborn as sns
from src import eda
sns.set_theme(style='whitegrid')

In [ ]:
df = dl.load_raw()
target = dl.find_target(df)
drops = dl.detect_droppable(df, target)
data = df.drop(columns=list(drops))
sat = (data[target] == config.POSITIVE_LABEL)

## Q1: Is the target balanced?

In [ ]:
ax = data[target].value_counts().plot.bar(rot=0, title='Satisfaction distribution')
ax.set_ylabel('Passengers')
plt.show()
print((data[target].value_counts(normalize=True) * 100).round(2))

## Q2: Does travel context matter?

In [ ]:
for col in ['Class', 'Type of Travel', 'Customer Type']:
    if col in data.columns:
        rate = data.groupby(col)[target].apply(lambda s: (s == config.POSITIVE_LABEL).mean() * 100)
        print(col + ':')
        print(rate.round(1).to_dict())
        print()

## Q3: How are delays distributed?

In [ ]:
for c in ['Departure Delay', 'Arrival Delay']:
    if c in data.columns:
        print(c, '| median', data[c].median(), '| p99', data[c].quantile(.99), '| max', data[c].max())

Both are extremely right-skewed: the median is 0 minutes while the maximum exceeds 1,500.

## Q4: Linear association with the target

In [ ]:
num = data.select_dtypes('number').copy()
num['__target__'] = sat.astype(int)
corr = num.corr()['__target__'].drop('__target__').sort_values(ascending=False)
print(corr.round(3))

No feature shows a strong linear correlation — which is why non-linear models beat Logistic Regression so decisively.

## Generate and save the full figure set

In [ ]:
made = eda.run_eda(data, target)
print('figures written:', made)